In [ ]:
import javabridge
import os
import glob
import pandas as pd
import pydot
from IPython.display import SVG

In [ ]:
tetrad_libdir = os.path.join(os.getcwd(), '../../','src', 'pycausal', 'lib')
for l in glob.glob(tetrad_libdir + os.sep + "*.jar"):
    print (l)
    javabridge.JARS.append(str(l))

In [ ]:
javabridge.start_vm(run_headless=True, max_heap_size = '100M')
javabridge.attach()

In [ ]:
def loadContinuousData(df, outputDataset = False):
    tetradData = None

    node_list = javabridge.JClassWrapper("java.util.ArrayList")()
    # load dataset
    for col in df.columns:
        nodi = javabridge.JClassWrapper("edu.cmu.tetrad.data.ContinuousVariable") (col)
        node_list.add(nodi)
    
    dataBox = javabridge.JClassWrapper("edu.cmu.tetrad.data.DoubleDataBox")(len(df.index),df.columns.size)
    
    for col in range(0,df.columns.size):
        for row in df.index:
            value = javabridge.make_instance("java/lang/Double", "(D)V", df.iloc[row,col])
            dataBox.set(row,col,value)
    
    tetradData = javabridge.JClassWrapper("edu.cmu.tetrad.data.BoxDataSet")(dataBox, node_list)
    
    if(not outputDataset):
        tetradData = javabridge.JClassWrapper('edu.cmu.tetrad.data.CovarianceMatrixOnTheFly')(tetradData)
        
    return tetradData

In [ ]:
datasets = javabridge.JClassWrapper('java.util.ArrayList')()
tetrad_images_dir = os.path.join(os.getcwd(), '../../', 'data', '9var')
for d in glob.glob(tetrad_images_dir + os.sep + "*.txt"):
    print (d)
    df = pd.read_table(d, sep="\t")
    tetradData = loadContinuousData(df)
    datasets.add(tetradData)

In [ ]:
penaltyDiscount = 4
maxDegree = 3
faithfulnessAssumed = True
verbose = True
# numBootstrap = 5
# ensembleMethod = 'Highest'

In [ ]:
score = javabridge.JClassWrapper('edu.cmu.tetrad.search.SemBicScoreImages')(datasets)
score.setPenaltyDiscount(penaltyDiscount)

In [ ]:
fges = javabridge.JClassWrapper('edu.cmu.tetrad.search.Fges')(score)
fges.setMaxDegree(maxDegree)
fges.setNumPatternsToStore(0)
fges.setFaithfulnessAssumed(faithfulnessAssumed)
fges.setVerbose(verbose)

In [ ]:
tetradGraph = fges.search()
tetradGraph

In [ ]:
tetradGraph.toString()

In [ ]:
tetradGraph.getNodeNames()

In [ ]:
tetradGraph.getEdges()

In [ ]:
graph = pydot.Dot(graph_type='digraph')

In [ ]:
n = tetradGraph.getNodeNames().toString()
n = n[1:len(n)-1]
n = n.split(",")
nodes = []
for i in range(0,len(n)):
    node = n[i]
    n[i] = node.strip()
    print (n[i])
    nodes.append(pydot.Node(n[i]))
    graph.add_node(nodes[i])

In [ ]:
def isNodeExisting(nodes,node):
    try:
        nodes.index(node)
        return True
    except IndexError:
        print ("Node {} does not exist!".format(node))
        return False

e = tetradGraph.getEdges().toString()
e = e[1:len(e)-1]
e = e.split(",")
for i in range(0,len(e)):
    e[i] = e[i].strip()
    token = e[i].split(" ")
    if(len(token) >= 3):
        src = token[0]
        arc = token[1]
        dst = token[2]
        if(isNodeExisting(n,src) and isNodeExisting(n,dst)):
            edge = pydot.Edge(nodes[n.index(src)],nodes[n.index(dst)])
            
            if(arc[0] != "-"):
                edge.set_dir("both")
            
            if(arc[0] == "o"):
                edge.set_arrowtail("odot")
            elif(arc[0] == "<"):
                edge.set_arrowtail("normal")
            
            if(arc[2] == "-"):
                edge.set_arrowhead("none")
            elif(arc[2] == "o"):
                edge.set_arrowhead("odot")
            else:
                edge.set_arrowhead("normal")
            
            graph.add_edge(edge)

In [ ]:
svg_str = graph.create_svg(prog='dot')
SVG(svg_str)

In [ ]:
javabridge.detach()
javabridge.kill_vm()